<a href="https://colab.research.google.com/github/aditibhat017-prog/cognitivefunction/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Digital Biomarkers of Cognitive Function in Older Adults

## Week 1: Data Exploration

### Research Question

Can wearable-derived activity and rest-activity features predict cognitive performance in adults aged 60 years and older?

### Hypothesis

Greater physical activity, greater activity intensity, and more stable rest-activity patterns will be associated with better cognitive performance.

### Dataset

NHANES 2011–2014

### Primary Outcome

Digit Symbol Substitution Test (DSST)

### Secondary Outcomes

- Animal Fluency
- CERAD word learning and delayed recall

## 1. Setup

This notebook will:

1. Download NHANES data
2. Load the cognitive, wearable, and demographic datasets
3. Inspect the data structure
4. Identify relevant variables
5. Perform initial quality control

In [1]:
import os
import pandas as pd
from pathlib import Path

In [2]:
# Create folders for our data
os.makedirs("data/raw", exist_ok=True)

print("Data folder created.")

Data folder created.


In [3]:
# NHANES files we need

files = {
    "CFQ_G.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/CFQ_G.xpt",
    "PAXDAY_G.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/PAXDAY_G.xpt",
    "DEMO_G.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DEMO_G.xpt",

    "CFQ_H.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/CFQ_H.xpt",
    "PAXDAY_H.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/PAXDAY_H.xpt",
    "DEMO_H.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/DEMO_H.xpt"
}

print(f"We will download {len(files)} files.")

We will download 6 files.


In [4]:
import requests

for filename, url in files.items():
    output_path = f"data/raw/{filename}"

    response = requests.get(url)

    with open(output_path, "wb") as f:
        f.write(response.content)

    print(f"Downloaded: {filename}")

Downloaded: CFQ_G.xpt
Downloaded: PAXDAY_G.xpt
Downloaded: DEMO_G.xpt
Downloaded: CFQ_H.xpt
Downloaded: PAXDAY_H.xpt
Downloaded: DEMO_H.xpt


In [5]:
from pathlib import Path

for file in Path("data/raw").glob("*.xpt"):
    print(file, f"{file.stat().st_size / 1_000_000:.2f} MB")

data/raw/CFQ_H.xpt 0.27 MB
data/raw/PAXDAY_G.xpt 6.49 MB
data/raw/PAXDAY_H.xpt 7.32 MB
data/raw/DEMO_H.xpt 3.83 MB
data/raw/DEMO_G.xpt 3.75 MB
data/raw/CFQ_G.xpt 0.26 MB


## 2. Load NHANES Data

In [6]:
# 2011–2012

cfq_g = pd.read_sas("data/raw/CFQ_G.xpt")
pax_g = pd.read_sas("data/raw/PAXDAY_G.xpt")
demo_g = pd.read_sas("data/raw/DEMO_G.xpt")

# 2013–2014

cfq_h = pd.read_sas("data/raw/CFQ_H.xpt")
pax_h = pd.read_sas("data/raw/PAXDAY_H.xpt")
demo_h = pd.read_sas("data/raw/DEMO_H.xpt")

In [7]:
print("2011–2012")
print("Cognitive:", cfq_g.shape)
print("Activity:", pax_g.shape)
print("Demographics:", demo_g.shape)

print("\n2013–2014")
print("Cognitive:", cfq_h.shape)
print("Activity:", pax_h.shape)
print("Demographics:", demo_h.shape)

2011–2012
Cognitive: (1687, 19)
Activity: (61168, 15)
Demographics: (9756, 48)

2013–2014
Cognitive: (1785, 19)
Activity: (69018, 15)
Demographics: (10175, 47)


In [8]:
cfq_h.head()

,SEQN,CFASTAT,CFALANG,CFDCCS,CFDCRNC,CFDCST1,CFDCST2,CFDCST3,CFDCSR,CFDCIT1,CFDCIT2,CFDCIT3,CFDCIR,CFDAPP,CFDARNC,CFDAST,CFDDPP,CFDDRNC,CFDDS
0,73557.0,1.0,1.0,1.0,NaN,4.0,7.0,9.0,7.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,1.0,NaN,16.0,1.0,NaN,54.0
1,73559.0,1.0,1.0,1.0,NaN,4.0,8.0,7.0,6.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,1.0,NaN,13.0,1.0,NaN,63.0
2,73561.0,1.0,1.0,1.0,NaN,6.0,10.0,10.0,10.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,1.0,NaN,19.0,1.0,NaN,59.0
3,73564.0,1.0,1.0,1.0,NaN,5.0,6.0,6.0,7.0,5.397605e-79,5.397605e-79,5.397605e-79,1.000000e+00,1.0,NaN,20.0,1.0,NaN,79.0
4,73567.0,1.0,1.0,1.0,NaN,6.0,9.0,10.0,10.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,1.0,NaN,15.0,1.0,NaN,30.0


In [9]:
cfq_h.columns.tolist()

['SEQN',
 'CFASTAT',
 'CFALANG',
 'CFDCCS',
 'CFDCRNC',
 'CFDCST1',
 'CFDCST2',
 'CFDCST3',
 'CFDCSR',
 'CFDCIT1',
 'CFDCIT2',
 'CFDCIT3',
 'CFDCIR',
 'CFDAPP',
 'CFDARNC',
 'CFDAST',
 'CFDDPP',
 'CFDDRNC',
 'CFDDS']

In [10]:
[c for c in cfq_h.columns if c.startswith("CFD")]

['CFDCCS',
 'CFDCRNC',
 'CFDCST1',
 'CFDCST2',
 'CFDCST3',
 'CFDCSR',
 'CFDCIT1',
 'CFDCIT2',
 'CFDCIT3',
 'CFDCIR',
 'CFDAPP',
 'CFDARNC',
 'CFDAST',
 'CFDDPP',
 'CFDDRNC',
 'CFDDS']

In [11]:
pax_h.head()

,SEQN,PAXDAYD,PAXDAYWD,PAXSSNDP,PAXMSTD,PAXTMD,PAXAISMD,PAXVMD,PAXMTSD,PAXWWMD,PAXSWMD,PAXNWMD,PAXUMD,PAXLXSD,PAXQFD
0,73557.0,b'1',b'3',5.397605e-79,b'16:30:00',450.0,915200.0,450.0,4576.422,2.370000e+02,2.200000e+01,174.0,17.0,1.470493e+05,5.397605e-79
1,73557.0,b'2',b'4',2.160000e+06,b' 0:00:00',1440.0,2989920.0,1440.0,12222.152,7.440000e+02,1.130000e+02,546.0,37.0,1.733408e+05,5.397605e-79
2,73557.0,b'3',b'5',9.072000e+06,b' 0:00:00',1440.0,3307840.0,1440.0,3158.922,1.470000e+02,5.397605e-79,1287.0,6.0,8.340733e+04,5.397605e-79
3,73557.0,b'4',b'6',1.598400e+07,b' 0:00:00',1440.0,6860400.0,1440.0,23.641,5.397605e-79,5.397605e-79,1439.0,1.0,5.397605e-79,5.397605e-79
4,73557.0,b'5',b'7',2.289600e+07,b' 0:00:00',1440.0,3968640.0,1440.0,8879.010,5.040000e+02,9.800000e+01,822.0,16.0,3.864985e+04,5.397605e-79


In [12]:
[c for c in pax_h.columns if c.startswith("PAX")]

['PAXDAYD',
 'PAXDAYWD',
 'PAXSSNDP',
 'PAXMSTD',
 'PAXTMD',
 'PAXAISMD',
 'PAXVMD',
 'PAXMTSD',
 'PAXWWMD',
 'PAXSWMD',
 'PAXNWMD',
 'PAXUMD',
 'PAXLXSD',
 'PAXQFD']

In [13]:
pax_h[
    [
        "SEQN",
        "PAXDAYD",
        "PAXMTSD",
        "PAXVMD",
        "PAXWWMD",
        "PAXSWMD"
    ]
].head()

,SEQN,PAXDAYD,PAXMTSD,PAXVMD,PAXWWMD,PAXSWMD
0,73557.0,b'1',4576.422,450.0,2.370000e+02,2.200000e+01
1,73557.0,b'2',12222.152,1440.0,7.440000e+02,1.130000e+02
2,73557.0,b'3',3158.922,1440.0,1.470000e+02,5.397605e-79
3,73557.0,b'4',23.641,1440.0,5.397605e-79,5.397605e-79
4,73557.0,b'5',8879.010,1440.0,5.040000e+02,9.800000e+01


In [14]:
demo_h.head()

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHREDU,DMDHRMAR,DMDHSEDU,WTINT2YR,WTMEC2YR,SDMVPSU,SDMVSTRA,INDHHIN2,INDFMIN2,INDFMPIR
0,73557.0,8.0,2.0,1.0,69.0,NaN,4.0,4.0,1.0,NaN,...,3.0,4.0,NaN,13281.237386,13481.042095,1.0,112.0,4.0,4.0,0.84
1,73558.0,8.0,2.0,1.0,54.0,NaN,3.0,3.0,1.0,NaN,...,3.0,1.0,1.0,23682.057386,24471.769625,1.0,108.0,7.0,7.0,1.78
2,73559.0,8.0,2.0,1.0,72.0,NaN,3.0,3.0,2.0,NaN,...,4.0,1.0,3.0,57214.803319,57193.285376,1.0,109.0,10.0,10.0,4.51
3,73560.0,8.0,2.0,1.0,9.0,NaN,3.0,3.0,1.0,119.0,...,3.0,1.0,4.0,55201.178592,55766.512438,2.0,109.0,9.0,9.0,2.52
4,73561.0,8.0,2.0,2.0,73.0,NaN,3.0,3.0,1.0,NaN,...,5.0,1.0,5.0,63709.667069,65541.871229,2.0,116.0,15.0,15.0,5.00


In [15]:
demo_h[
    [
        "SEQN",
        "RIDAGEYR",
        "RIAGENDR",
        "DMDEDUC2"
    ]
].head()

,SEQN,RIDAGEYR,RIAGENDR,DMDEDUC2
0,73557.0,69.0,1.0,3.0
1,73558.0,54.0,1.0,3.0
2,73559.0,72.0,1.0,4.0
3,73560.0,9.0,1.0,NaN
4,73561.0,73.0,2.0,5.0


## 3. Inspecting Cognitive Variables

In [16]:
# Inspect the primary and secondary cognitive outcomes

cognitive_vars = [
    "SEQN",
    "CFDDS",
    "CFDAST",
    "CFDCSR",
    "CFDCST1",
    "CFDCST2",
    "CFDCST3"
]

cfq_h[cognitive_vars].describe()

,SEQN,CFDDS,CFDAST,CFDCSR,CFDCST1,CFDCST2,CFDCST3
count,1785.000000,1.592000e+03,1661.000000,1.672000e+03,1.681000e+03,1.678000e+03,1.674000e+03
mean,78665.612885,4.595101e+01,16.427453,6.084330e+00,4.759667e+00,6.802741e+00,7.682198e+00
std,2965.804479,1.723167e+01,5.551864,2.392026e+00,1.746238e+00,1.955636e+00,1.927702e+00
min,73557.000000,5.397605e-79,3.000000,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
25%,76149.000000,3.300000e+01,12.000000,5.000000e+00,4.000000e+00,6.000000e+00,7.000000e+00
50%,78640.000000,4.600000e+01,16.000000,6.000000e+00,5.000000e+00,7.000000e+00,8.000000e+00
75%,81269.000000,5.800000e+01,20.000000,8.000000e+00,6.000000e+00,8.000000e+00,9.000000e+00
max,83724.000000,1.050000e+02,39.000000,1.000000e+01,1.000000e+01,1.000000e+01,1.000000e+01


In [17]:
cfq_h[cognitive_vars].isna().sum()

,0
SEQN,0
CFDDS,193
CFDAST,124
CFDCSR,113
CFDCST1,104
CFDCST2,107
CFDCST3,111


## 4. Inspecting Wearable Variables

In [18]:
wearable_vars = [
    "SEQN",
    "PAXDAYD",
    "PAXMTSD",
    "PAXVMD",
    "PAXWWMD",
    "PAXSWMD",
    "PAXNWMD",
    "PAXUMD",
    "PAXQFD"
]

pax_h[wearable_vars].describe()

,SEQN,PAXMTSD,PAXVMD,PAXWWMD,PAXSWMD,PAXNWMD,PAXUMD,PAXQFD
count,69018.000000,6.901800e+04,6.901800e+04,6.901800e+04,6.901800e+04,6.901800e+04,6.901800e+04,6.901800e+04
mean,78667.985540,1.126228e+04,1.274297e+03,6.804351e+02,3.770694e+02,1.742785e+02,4.251432e+01,4.556159e+00
std,2935.050609,7.074941e+03,3.380953e+02,3.500921e+02,2.190723e+02,3.642156e+02,2.689863e+01,7.068316e+01
min,73557.000000,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
25%,76129.000000,5.669873e+03,1.434000e+03,4.110000e+02,2.430000e+02,5.397605e-79,2.400000e+01,5.397605e-79
50%,78703.000000,1.157286e+04,1.440000e+03,8.320000e+02,4.180000e+02,5.397605e-79,4.000000e+01,5.397605e-79
75%,81199.000000,1.633570e+04,1.440000e+03,9.440000e+02,5.100000e+02,1.040000e+02,5.900000e+01,5.397605e-79
max,83731.000000,4.494209e+04,1.440000e+03,1.438000e+03,1.440000e+03,1.440000e+03,2.510000e+02,3.110000e+03


In [19]:
pax_h[wearable_vars].isna().sum()

,0
SEQN,0
PAXDAYD,0
PAXMTSD,0
PAXVMD,0
PAXWWMD,0
PAXSWMD,0
PAXNWMD,0
PAXUMD,0
PAXQFD,0


In [20]:
pax_h[wearable_vars].dtypes

,0
SEQN,float64
PAXDAYD,object
PAXMTSD,float64
PAXVMD,float64
PAXWWMD,float64
PAXSWMD,float64
PAXNWMD,float64
PAXUMD,float64
PAXQFD,float64


In [21]:
pax_h["PAXMTSD"].sort_values().head(20)

,PAXMTSD
39766,5.397605e-79
2957,5.397605e-79
2958,5.397605e-79
35966,5.397605e-79
35967,5.397605e-79
60083,5.397605e-79
2690,5.397605e-79
24751,5.397605e-79
15846,5.397605e-79
15847,5.397605e-79


In [22]:
pax_h["PAXMTSD"].sort_values(ascending=False).head(20)

,PAXMTSD
68461,44942.093
7431,43921.869
36422,41011.590
9944,40990.555
28941,40766.416
11658,40734.680
56341,40572.940
32241,40110.740
56807,39872.828
45571,39565.446


In [23]:
pax_h["PAXVMD"].value_counts().sort_index().head(20)

,count
PAXVMD,
5.397605e-79,35
1.000000e+00,3
2.000000e+00,1
3.000000e+00,1
4.000000e+00,1
6.000000e+00,2
1.000000e+01,2
1.100000e+01,1
3.300000e+01,1


In [24]:
pax_h["PAXVMD"].describe()

,PAXVMD
count,6.901800e+04
mean,1.274297e+03
std,3.380953e+02
min,5.397605e-79
25%,1.434000e+03
50%,1.440000e+03
75%,1.440000e+03
max,1.440000e+03


In [25]:
days_per_person = pax_h.groupby("SEQN")["PAXDAYD"].nunique()

days_per_person.describe()

,PAXDAYD
count,7776.000000
mean,8.875772
std,0.795324
min,1.000000
25%,9.000000
50%,9.000000
75%,9.000000
max,9.000000


In [26]:
days_per_person.value_counts().sort_index()

,count
PAXDAYD,
1,7
2,37
3,30
4,31
5,30
6,31
7,30
8,43
9,7537


In [27]:
cfq_h[cognitive_vars].isna().sum()

,0
SEQN,0
CFDDS,193
CFDAST,124
CFDCSR,113
CFDCST1,104
CFDCST2,107
CFDCST3,111


In [28]:
pax_h[wearable_vars].describe()

,SEQN,PAXMTSD,PAXVMD,PAXWWMD,PAXSWMD,PAXNWMD,PAXUMD,PAXQFD
count,69018.000000,6.901800e+04,6.901800e+04,6.901800e+04,6.901800e+04,6.901800e+04,6.901800e+04,6.901800e+04
mean,78667.985540,1.126228e+04,1.274297e+03,6.804351e+02,3.770694e+02,1.742785e+02,4.251432e+01,4.556159e+00
std,2935.050609,7.074941e+03,3.380953e+02,3.500921e+02,2.190723e+02,3.642156e+02,2.689863e+01,7.068316e+01
min,73557.000000,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
25%,76129.000000,5.669873e+03,1.434000e+03,4.110000e+02,2.430000e+02,5.397605e-79,2.400000e+01,5.397605e-79
50%,78703.000000,1.157286e+04,1.440000e+03,8.320000e+02,4.180000e+02,5.397605e-79,4.000000e+01,5.397605e-79
75%,81199.000000,1.633570e+04,1.440000e+03,9.440000e+02,5.100000e+02,1.040000e+02,5.900000e+01,5.397605e-79
max,83731.000000,4.494209e+04,1.440000e+03,1.438000e+03,1.440000e+03,1.440000e+03,2.510000e+02,3.110000e+03


In [29]:
pax_h["PAXMTSD"].sort_values().head(20)

,PAXMTSD
39766,5.397605e-79
2957,5.397605e-79
2958,5.397605e-79
35966,5.397605e-79
35967,5.397605e-79
60083,5.397605e-79
2690,5.397605e-79
24751,5.397605e-79
15846,5.397605e-79
15847,5.397605e-79


In [30]:
pax_h["PAXMTSD"].sort_values(ascending=False).head(20)

,PAXMTSD
68461,44942.093
7431,43921.869
36422,41011.590
9944,40990.555
28941,40766.416
11658,40734.680
56341,40572.940
32241,40110.740
56807,39872.828
45571,39565.446


In [31]:
days_per_person.value_counts().sort_index()

,count
PAXDAYD,
1,7
2,37
3,30
4,31
5,30
6,31
7,30
8,43
9,7537
